# 04 - Preview Analytics
**Objetivos:**  
Realizar análises exploratórias sobre o dataset já tratado e com features criadas, buscando identificar padrões de comparecimento, comportamento por tipo de pagamento e evolução dos pacientes ao longo do tempo. 

**Análises realizadas:**
- Distribuição dos status
- Distribuição dos tipos de pagamento
- Comparação: Status x Tipo de pagament o
- Comparação: Status x Paciente
- Comparação: Paciente x Ano

In [2]:
# =============================
# Carregar CSV features
# =============================

import pandas as pd

df = pd.read_csv("../data/processed/appointments_feature.csv", parse_dates=['appointment_date'])
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1188 entries, 0 to 1187
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   appointment_id      1188 non-null   str           
 1   appointment_date    1188 non-null   datetime64[us]
 2   appointment_status  1188 non-null   str           
 3   patient_id          1188 non-null   str           
 4   appointment_price   1188 non-null   float64       
 5   payment_type        1188 non-null   str           
 6   sessions_per_week   1188 non-null   int64         
 7   attended_flag       1188 non-null   int64         
 8   is_package          1188 non-null   int64         
 9   month               1188 non-null   str           
 10  month_date          1188 non-null   str           
 11  weekday             1188 non-null   str           
 12  weekday_num         1188 non-null   int64         
dtypes: datetime64[us](1), float64(1), int64(4), str(7)
memory u

### Distribuição por Status
Essa análise tem como objetivo quantificar os atendimentos realizados e as faltas, apresentando também sua distribuição percentual.

**Insight:**  
Permite entender a taxa geral de comparecimento aos atendimentos no consultório.

In [3]:
status_count = df['appointment_status'].value_counts().reset_index(name='count')

status_count['proportion'] = status_count['count']\
                            .div(status_count['count'].sum())\
                            .mul(100)\
                            .round(2)

status_count.sort_values('count', ascending=False)
status_count.style.format({'proportion': '{:.2f}%'})

,appointment_status,count,proportion
0,Attended,961,80.89%
1,Missed,227,19.11%


### Distribuição por tipo de pagamento
Analisa a proporção de atendimentos entre pagamento por sessão e paconte mensal.

**Insight:**  
Mostra a quantidade e a proporção de cada tipo de pagamento no dataset.

In [4]:
payment_count = df['payment_type'].value_counts().reset_index(name='count')

payment_count['proportion'] = payment_count['count']\
                            .div(payment_count['count'].sum())\
                            .mul(100)\
                            .round(2)

payment_count.sort_values('count', ascending=False)
payment_count.style.format({'proportion': '{:.2f}%'})

,payment_type,count,proportion
0,PerSession,603,50.76%
1,MonthlyPackage,585,49.24%


### Status x Tipo de pagamento
Comparar presença e falta entre os métodos de pagamentos do consultório.

**Insight:**  
A mudança do método de pagamento *PerSession* para *MonthlyPackage* está associada a uma redução superior a 50% na taxa de faltas, sugerindo maior compromentimento do paciente.

In [5]:
comparacao_pacote = pd.crosstab(df['payment_type'],\
                                df['appointment_status'],\
                                normalize='index')

comparacao_pacote = comparacao_pacote.reset_index().style.format({'Attended': '{:.2%}',
                                                                  'Missed': '{:.2%}'})

comparacao_pacote

appointment_status,payment_type,Attended,Missed
0,MonthlyPackage,87.52%,12.48%
1,PerSession,74.46%,25.54%


### Paciente x Status
Essa análise tem como objetivo verificar a taxa de comparecimento de cada paciente.  

**Insight:**  
A análise mostra que a maioria dos pacientes possuem uma frequência superior a 80%.  
Um paciente apresentou frequência significamente inferior, podendo ser considerado um *outlier*.

In [6]:
attended_percent = pd.crosstab(df['patient_id'],\
                                df['appointment_status'],\
                                normalize='index')\
                                .sort_values('Attended', ascending=False)

attended_percent = attended_percent.reset_index().style.format({'Attended': '{:.2%}',
                                                                'Missed': '{:.2%}'})
attended_percent

appointment_status,patient_id,Attended,Missed
0,P007,94.23%,5.77%
1,P002,92.31%,7.69%
2,P006,91.45%,8.55%
3,P008,89.42%,10.58%
4,P011,88.46%,11.54%
5,P013,86.54%,13.46%
6,P012,84.62%,15.38%
7,P010,82.20%,17.80%
8,P004,81.73%,18.27%
9,P005,73.56%,26.44%


### Paciente x Ano
Essa análise tem como objetivo verificar a taxa de faltas dos pacientes em 2022 e em 2023 e calcular qual foi a taxa de variação entre os anos.

**Insight:**
* 6 de 7 pacientes com dados em ambos os anos apresentaram melhora
* Pacientes que iniciaram diretamente no pacote mensal já mostraram taxas de faltas reduzidas
* O paciente P004 apresentou piora, associada a questões de saúde.

In [7]:
comparacao_anual = df.groupby(['patient_id',
                               df['appointment_date'].dt.year])['appointment_status']\
                                .apply(lambda x: (x == 'Missed').mean())\
                                .unstack()
                                
comparacao_anual.columns = ['miss_rate_22','miss_rate_23']

comparacao_anual['melhora'] = comparacao_anual['miss_rate_22'] - comparacao_anual['miss_rate_23']

comparacao_anual.sort_values('melhora', ascending=False)\
                .reset_index()\
                .style.format({'miss_rate_22': '{:.2%}',
                               'miss_rate_23': '{:.2%}',
                               'melhora': '{:.2%}'},
                                na_rep="-")

,patient_id,miss_rate_22,miss_rate_23,melhora
0,P005,35.58%,17.31%,18.27%
1,P010,26.42%,10.77%,15.65%
2,P009,34.78%,19.23%,15.55%
3,P008,17.31%,3.85%,13.46%
4,P007,9.62%,1.92%,7.69%
5,P006,10.77%,5.77%,5.00%
6,P004,15.38%,21.15%,-5.77%
7,P001,61.54%,-,-
8,P002,7.69%,-,-
9,P003,26.92%,-,-
